# Biclustering and multi-clustering: the schematic of figure 21.21

Figure 21.21 of the book draws two ways of clustering the rows and columns of a $6 \times 6$ data matrix. Each cell
is labelled $r.c$: the cell belongs to row cluster $r$ within column group $c$, and its colour shows the block.
The book has the figure as an image only; this notebook redraws it from the cluster labels.

The difference between the panels is how many partitions of the rows there are. In a biclustering
([biclustering_irm.ipynb](biclustering_irm.ipynb), figure 21.20) the columns are grouped and the rows are partitioned
once, for all column groups. In a multi-clustering, or nested partition model, every column group (a *view*) has a
partition of the rows of its own, so a row can be grouped with different rows depending on which features we look
at; this is CrossCat ([crosscat_animals.ipynb](crosscat_animals.ipynb), figure 21.22).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

try:
    import probml_utils as pml
except ModuleNotFoundError:
    %pip install -qq git+https://github.com/probml/probml-utils.git
    import probml_utils as pml

## The two panels

The labels and colours of the book's figure. The six columns form three groups: columns 1-2, column 3 and
columns 4-6.

In [ ]:
panels = {
    "A": [
        ["1.1", "1.1", "1.2", "1.3", "1.3", "1.3"],
        ["1.1", "1.1", "1.2", "1.3", "1.3", "1.3"],
        ["1.1", "1.1", "1.2", "1.3", "1.3", "1.3"],
        ["2.1", "2.1", "2.2", "2.3", "2.3", "2.3"],
        ["2.1", "2.1", "3.2", "2.3", "2.3", "2.3"],
        ["2.1", "2.1", "3.2", "2.3", "2.3", "2.3"],
    ],
    "B": [
        ["1.1", "1.1", "1.2", "1.3", "1.3", "1.3"],
        ["1.1", "1.1", "1.2", "1.3", "1.3", "1.3"],
        ["1.1", "1.1", "2.2", "1.3", "1.3", "1.3"],
        ["2.1", "2.1", "2.2", "1.3", "1.3", "1.3"],
        ["2.1", "2.1", "3.2", "1.3", "1.3", "1.3"],
        ["2.1", "2.1", "3.2", "2.3", "2.3", "2.3"],
    ],
}
# the book's colours (2.2 and 3.2 share orange there too)
colors = {
    "1.1": "#ee2a2f", "2.1": "#6ccde0",
    "1.2": "#7b52a1", "2.2": "#f6941f", "3.2": "#f6941f",
    "1.3": "#6bbd4a", "2.3": "#f7e81b",
}


def draw_panel(ax, labels, title):
    labels = np.array(labels)
    n_rows, n_cols = labels.shape
    for i in range(n_rows):
        for j in range(n_cols):
            ax.add_patch(Rectangle((j, n_rows - 1 - i), 1, 1, facecolor=colors[labels[i, j]], edgecolor="#333333", lw=1))
            ax.text(j + 0.08, n_rows - 1 - i + 0.5, labels[i, j], va="center", fontsize=11, color="#222222")
    ax.set_xlim(0, n_cols)
    ax.set_ylim(0, n_rows)
    ax.set_aspect(0.33)
    ax.axis("off")
    ax.set_title(title, loc="left", fontsize=14, fontweight="bold")


for name, labels in panels.items():
    fig, ax = plt.subplots(figsize=(8, 2.8))
    draw_panel(ax, labels, f"({name.lower()})")
    fig.tight_layout()
    pml.savefig(f"multiclustering_schematic_{name}.pdf")
    plt.show()

## The row partition in each column group

Reading the labels column group by column group gives the partition of the six rows that each group uses.

In [ ]:
groups = {1: [0, 1], 2: [2], 3: [3, 4, 5]}  # column group -> columns


def row_partitions(labels):
    labels = np.array(labels)
    parts = {}
    for c, cols in groups.items():
        assert all(len(set(labels[i, cols])) == 1 for i in range(len(labels))), "a row splits inside a column group"
        clusters = {}
        for i, lab in enumerate(labels[:, cols[0]]):
            clusters.setdefault(lab.split(".")[0], []).append(i + 1)
        parts[c] = list(clusters.values())
    return parts


for name, labels in panels.items():
    parts = row_partitions(labels)
    distinct = {str(p) for p in parts.values()}
    print(f"Panel {name}: {len(distinct)} distinct row partition(s)")
    for c, p in parts.items():
        print(f"  column group {c}: " + " | ".join("rows " + ",".join(map(str, block)) for block in p))

In panel (b) the three column groups partition the rows in three different ways: rows 3 and 4 are together
for column 3 but apart for columns 1-2, and row 6 is on its own only for columns 4-6. A single partition of the
rows, as in biclustering, cannot express this; giving each view its own partition, as CrossCat does, can.

In panel (a), columns 1-2 and 4-6 share the partition {1,2,3 | 4,5,6}, and column 3 splits rows 4-6 further
({4 | 5,6}): it is drawn with the same labels as the book's figure.